# 8 · From numpy to Daft pipelines

The tiny tier loads everything into a numpy array. That doesn't scale. Here we
express the data pipeline as a lazy **Daft** dataframe over the Lance dataset —
the same code runs on a cluster against `bdd-full` (Chapter 9).

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


### Read Lance into Daft and push the split filter down

In [2]:
import daft, lance
tbl = lance.dataset(str(DATA)).to_table().drop(["image"])   # metadata columns
df = daft.from_arrow(tbl)
print("schema:"); print(df.schema())
train = df.where(df["split"] == "train")
print("train rows:", train.count_rows())
(train.groupby("weather").agg(daft.col("id").count().alias("n"))
      .sort("weather").show())

weatherString,nUInt64
clear,670
foggy,196
overcast,477
rainy,392
snowy,365


### Stream cue features into the dataframe and aggregate per slice

In [3]:
# In production the embedding is a Daft UDF over the image column (sketch below);
# here we compute it with the harness helper and attach it as columns.
from harness.mining import embed
x, y, meta = load_split(DATA, "train")
f = embed(x)   # (N, 4): R,G,B colour ratio + contrast
feat = daft.from_pydict({
    "weather": meta["weather"], "r": f[:, 0], "g": f[:, 1], "b": f[:, 2],
})
(feat.groupby("weather")
     .agg(daft.col("r").mean().alias("R"), daft.col("b").mean().alias("B"))
     .sort("weather").show())
print("Fog frames sit apart in colour-ratio space -> that's what mining exploits.")

weatherString,RFloat64,BFloat64
clear,0.33310501326375935,0.33474544695953823
foggy,0.3329277038574219,0.33470893392757495
overcast,0.33403878841760026,0.3335465725113011
rainy,0.33319535547373247,0.33435696971659756
snowy,0.3329415935359589,0.33471606528922304


Fog frames sit apart in colour-ratio space -> that's what mining exploits.


```python
# production: decode + featurize as a streamed UDF, never materializing the set
@daft.udf(return_dtype=daft.DataType.fixed_size_list(daft.DataType.float32(), 4))
def colour_signature(images): ...
df = daft.read_lance("data/bdd-full.lance").with_column("embed", colour_signature(df["image"]))
```
The query plan, the filter pushdown, and the per-slice aggregation are identical
whether the data is 3K rows on a laptop or 100K on a Ray cluster.